<a href="https://colab.research.google.com/github/vishalsd16/Assignment_AI-ML/blob/main/Next_word_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing required libraries:

In [1]:
import urllib.request
import re
import math
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# =====================================================
# CONFIG
# =====================================================

URL = "https://www.gutenberg.org/files/1661/1661-0.txt"

FILE_PATH = "data/sherlock.txt"

EPOCHS = 30
BATCH_SIZE = 32

# =====================================================
# DOWNLOAD DATA
# =====================================================

def download_data():

    try:
        with open(FILE_PATH, "r", encoding="utf-8"):
            print("Dataset already exists.")

    except FileNotFoundError:

        print("Downloading dataset...")

        urllib.request.urlretrieve(URL, FILE_PATH)

        print("Download completed.")

# =====================================================
# CLEAN TEXT
# =====================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s']", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text

# =====================================================
# GENERATE TEXT
# =====================================================

def generate_text(seed_text, next_words, model, tokenizer, max_len):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_len - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)

        predicted_word_index = np.argmax(predicted_probs)

        output_word = ""

        for word, index in tokenizer.word_index.items():

            if index == predicted_word_index:

                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

# =====================================================
# MAIN
# =====================================================

download_data()

with open(FILE_PATH, "r", encoding="utf-8") as file:

    text = file.read()

text = clean_text(text)

# Smaller subset helps overfit easier
text = text[:800000]

tokenizer = Tokenizer()

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary Size:", total_words)

token_list = tokenizer.texts_to_sequences([text])[0]

# =====================================================
# CREATE SEQUENCES
# =====================================================

input_sequences = []

sequence_length = 20

for i in range(sequence_length, len(token_list)):

    n_gram_sequence = token_list[i-sequence_length:i+1]

    input_sequences.append(n_gram_sequence)

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]

y = input_sequences[:, -1]

# =====================================================
# MODEL
# =====================================================

model = Sequential()

model.add(
    Embedding(
        input_dim=total_words,
        output_dim=100
    )
)

model.add(
    LSTM(150)
)

model.add(
    Dense(
        total_words,
        activation='softmax'
    )
)

# Explicitly build model
model.build(input_shape=(None, sequence_length))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# =====================================================
# TRAIN
# =====================================================

early_stop = EarlyStopping(
    monitor='loss',
    patience=3
)

history = model.fit(
    X,
    y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

# =====================================================
# FINAL METRICS
# =====================================================

final_accuracy = history.history['accuracy'][-1]

final_loss = history.history['loss'][-1]

perplexity = math.exp(final_loss)

print("\nFINAL RESULTS")
print("=" * 40)

print(f"Training Accuracy: {final_accuracy:.4f}")

print(f"Loss: {final_loss:.4f}")

print(f"Perplexity: {perplexity:.2f}")

# =====================================================
# SAVE MODEL
# =====================================================

model.save("next_word_model.keras")

print("\nModel saved successfully.")

# =====================================================
# TEXT GENERATION
# =====================================================

seeds = [
    "sherlock holmes",
    "it was",
    "i think",
]

print("\nGENERATED TEXT")
print("=" * 40)

for seed in seeds:

    generated = generate_text(
        seed,
        15,
        model,
        tokenizer,
        sequence_length + 1
    )

    print(f"\nSeed: {seed}")

    print(f"Generated: {generated}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/sherlock.txt'

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import pickle
import numpy as np
import os

In [2]:
from google.colab import files
uploaded = files.upload()

Load and Pre-Process the data

In [4]:
file = open("/content/sample_data/holmes.txt", "r", encoding = "utf8" )

#store file in list
lines = []
for i in file:
     lines.append(i)

#Convert list to string
data = ""
for i in lines:
    data = '  '. join(lines)

#replace unnecessary stuff with space
data = data.replace('\n', '').replace('\r', '').replace('\ufeff', '').replace('“','').replace('”','')  #new line, carriage return, unicode character --> replace by space

#remove unnecessary spaces
data = data.split()
data = ' '.join(data)
data[:500]


'The Project Gutenberg eBook of The Adventures of Sherlock Holmes, by Arthur Conan Doyle This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org. If you are not located in the United States, you will have to check the laws of the country where you are lo'

In [5]:
len(data)

573200

Apply tokenization and some other changes

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

#saving the tokenizer for predict function
pickle.dump(tokenizer, open('token.pkl','wb'))

sequence_data = tokenizer.texts_to_sequences([data])[0]
sequence_data[:15]

[1, 140, 294, 857, 5, 1, 1060, 5, 127, 33, 46, 590, 2586, 2587, 27]

In [7]:
len(sequence_data)

108857

In [8]:
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

8605


In [9]:
sequences = []

for i in range (3, len(sequence_data)):
  words = sequence_data[i-3:i+1]
  sequences.append(words)

print("The Length if sequences are:" , len(sequences))
sequences = np.array(sequences)
sequences[:10]

The Length if sequences are: 108854


array([[   1,  140,  294,  857],
       [ 140,  294,  857,    5],
       [ 294,  857,    5,    1],
       [ 857,    5,    1, 1060],
       [   5,    1, 1060,    5],
       [   1, 1060,    5,  127],
       [1060,    5,  127,   33],
       [   5,  127,   33,   46],
       [ 127,   33,   46,  590],
       [  33,   46,  590, 2586]])

In [10]:
X = []
y = []

for i in sequences:
  X.append(i[0:3])
  y.append(i[3])

X = np.array(X)
y = np.array(y)

In [11]:
print("Data:", X[:10])
print("Response:", y[:10])

Data: [[   1  140  294]
 [ 140  294  857]
 [ 294  857    5]
 [ 857    5    1]
 [   5    1 1060]
 [   1 1060    5]
 [1060    5  127]
 [   5  127   33]
 [ 127   33   46]
 [  33   46  590]]
Response: [ 857    5    1 1060    5  127   33   46  590 2586]


In [12]:
y = to_categorical(y, num_classes=vocab_size)
y[:5]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

Creating the model

In [24]:
model = Sequential()
model.add(Embedding(vocab_size, 10, input_length=3))
model.add(LSTM(500, return_sequences=True))
model.add(LSTM(500))
model.add(Dense(1000, activation="relu"))
model.add(Dense(vocab_size, activation="softmax"))
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Plot the model

In [19]:
from tensorflow import keras
from keras.utils.vis_utils import plot_model

keras.utils.plot_model(model, to_file='plot.png', show_layer_names=True)

ModuleNotFoundError: No module named 'keras.utils.vis_utils'

Train the model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint("next_word.h5", monitor='loss', verbise=1, save_best_only=True)
model.compile(loss="categorical_crossentropy", optimizer=Adam(learning_rate=0.001))
model.fit(X, y, epochs=20, batch_size=64, callbacks=[checkpoint])

Epoch 1/20
1126/1126 [==============================] - 33s 21ms/step - loss: 6.7736
Epoch 2/20
1126/1126 [==============================] - 18s 16ms/step - loss: 6.2607
Epoch 3/20
1126/1126 [==============================] - 18s 16ms/step - loss: 5.8552
Epoch 4/20
1126/1126 [==============================] - 19s 17ms/step - loss: 5.5418
Epoch 5/20
1126/1126 [==============================] - 18s 16ms/step - loss: 5.2844
Epoch 6/20
1126/1126 [==============================] - 18s 16ms/step - loss: 5.0524
Epoch 7/20
1126/1126 [==============================] - 18s 16ms/step - loss: 4.8328
Epoch 8/20
1126/1126 [==============================] - 18s 16ms/step - loss: 4.6022
Epoch 9/20
1126/1126 [==============================] - 18s 16ms/step - loss: 4.3600
Epoch 10/20
1126/1126 [==============================] - 18s 16ms/step - loss: 4.1134
Epoch 11/20
1126/1126 [==============================] - 18s 16ms/step - loss: 3.8681
Epoch 12/20
1126/1126 [==============================] - 18s 16

Lets Predict

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
import pickle

#Load the model and tokenizer
model = load_model('next_word.h5')
tokenizer = pickle.load(open('token.pkl', 'rb'))

def Predict_Next_Words(model, tokenizer, text):

  sequence = tokenizer.texts_to_sequences([text])
  sequence = np.array(sequence)
  preds = np.argmax(model.predict(sequence))
  predict_word = ""

  for key, value in tokenizer.word_index.items():
    if value == preds:
      predicted_word = key
      break

  print(predicted_word)
  return predicted_word

In [ ]:
while(True):
  text = input("Enter your line:")

  if text == "0":
    print("Execution completed....")
    break

  else:
    try:
      text = text.split(" ")
      text = text[-3:]
      print(text)

      Predict_Next_Words(model, tokenizer, text)


    except Exception as e:
       print("Error occured: ",e)
       continue

Enter your line:The Project Gutenberg
['The', 'Project', 'Gutenberg']
1/1 [==============================] - 1s 1s/step
tm
Enter your line:The Project Gutenberg
['The', 'Project', 'Gutenberg']
1/1 [==============================] - 0s 20ms/step
tm
Enter your line:The Project Gutenberg
['The', 'Project', 'Gutenberg']
1/1 [==============================] - 0s 22ms/step
tm
Enter your line:The Project Gutenberg eBook of
['Gutenberg', 'eBook', 'of']
1/1 [==============================] - 0s 21ms/step
the
Enter your line:He was quite
['He', 'was', 'quite']
1/1 [==============================] - 0s 22ms/step
a
Enter your line:however, it may all come to
['all', 'come', 'to']
1/1 [==============================] - 0s 21ms/step
the
